# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook is an interactive guide for exploring the FAIR<sup>2</sup> dataset: **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells**.

`mlcroissant` helps you load, explore, and analyze Croissant-formatted datasets including discovery of fields/columns and automated DataFrame loading.

### Dataset Source
- [Croissant schema (JSON-LD)](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) ([permalink](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json))
- FAIR<sup>2</sup> dataset title: **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells**

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # This is an object (not a dict!)

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}")

## 2. Data Overview
Review available record sets (tables), fields, and their `@id`s in the dataset. The Croissant schema allows us to examine available record sets programmatically using the Dataset object.


In [ ]:
# List available record sets and their fields/columns
print("Record Sets in dataset (by @id):\n")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"@id: {rs.id}")
    print(f"  Name: {rs.name}")
    if hasattr(rs, 'description'):
        print(f"  Description: {rs.description}")
    # List fields/columns
    print("  Fields/Columns (@id):")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, type: {field.data_type if hasattr(field, 'data_type') else 'unknown'})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

We will select the main record set containing protein measurements for demonstration. **Edit the `record_sets_to_load` variable to load others**.

In [ ]:
# Prepare list of record sets to analyze (use @ids from previous cell output)
# You may change these to load other tables as needed.
record_sets_to_load = [rs.id for rs in dataset.record_sets]
print(f"Available record set IDs: {record_sets_to_load}")

dataframes = {}
for record_set_id in record_sets_to_load:
    print(f"Loading data from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(), "\n")
    else:
        print("No records found for this record set.\n")

# Select a main record set for analysis (by @id)
main_record_set_id = record_sets_to_load[0] if record_sets_to_load else None
print(f"Main record set used for analysis: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Let's process the main record set's DataFrame. We'll select a numeric field for filtering and normalization; update the field `@id` as desired.

In [ ]:
# Choose a numeric field by its @id (replace as appropriate from the data overview)
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Heuristic: pick a likely numeric field
    possible_numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'iufc' and df[col].notna().sum() > 0]
    if not possible_numeric_fields:
        # Try to detect columns that are numbers but loaded as object
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if df[col].dtype.kind in 'iufc' and df[col].notna().sum() > 0:
                    possible_numeric_fields.append(col)
            except Exception:
                continue
    print(f"Detected numeric fields: {possible_numeric_fields}")
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else None
    if numeric_field:
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to group by a string/categorical field
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].nunique() < 20:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No main record set data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships. Let's plot the distribution of the selected numeric field and (if there is a grouping) a bar chart.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if main_record_set_id and main_record_set_id in dataframes and 'numeric_field' in locals() and numeric_field is not None:
    # Distribution
    sns.histplot(df[numeric_field].dropna(), kde=True, color='skyblue')
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouped
    if 'group_field' in locals() and group_field is not None:
        meanvals = df.groupby(group_field)[numeric_field].mean().sort_values()
        meanvals.plot(kind='bar', color='teal')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion

- We loaded and explored the **FAIR<sup>2</sup> mass spectrometry** dataset using the `mlcroissant` library.
- Using the dataset's Croissant schema, we programmatically identified record sets, fields, and types, using their `@id` values.
- We extracted the main record set into a DataFrame, performed filtering and normalization on a numeric field, and performed basic grouping operations.
- Simple data visualizations illustrated the dataset structure and value distribution.

**Next steps:**
- Review field definitions and units in the Croissant metadata for field-specific analysis.
- Apply custom filters or advanced analytics based on your research needs.
- Use the `mlcroissant` documentation to extract, transform, and annotate data with full semantic provenance.
